In [1]:
import tensorflow as tf
import numpy as np

### 1) Loading data

In [2]:
landmarks_data = np.load('../../../data/divided/landmarks_dataset.npz')

# Extract the individual arrays using their keys
X_train_landmarks = landmarks_data['X_train']
y_train_landmarks = landmarks_data['y_train']
X_test_landmarks = landmarks_data['X_test']
y_test_landmarks = landmarks_data['y_test']

images_data = np.load('../../../data/divided/images_dataset.npz')

# Extract the individual arrays using their keys
X_train_images = images_data['X_train']
y_train_images = images_data['y_train']
X_test_images = images_data['X_test']
y_test_images = images_data['y_test']


### 2) Defining Mobilenet model architecture

In [3]:
# Define MobileNet Model
def build_mobilenet(input_shape, num_classes):
    # Base MobileNetV2 model without the top dense layers
    base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape,
                                                   include_top=False,
                                                   weights=None)  # You can use pre-trained weights if needed

    # Adding custom top layers for classification
    model = tf.keras.Sequential([
        base_model,  # Base MobileNetV2
        tf.keras.layers.GlobalAveragePooling2D(),  # Converts to 1D tensor
        tf.keras.layers.Dense(64, activation='relu'),  # Custom dense layer
        tf.keras.layers.Dense(num_classes, activation='softmax')  # Output layer
    ])

    return model

### 3) Creating MobileNet model trained on images

In [4]:
from keras.applications import MobileNet
from keras.models import Model
from keras.layers import GlobalAveragePooling2D, Dense, Dropout

def build_improved_mobilenet(input_shape, num_classes):
    base_model = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)  # Additional dense layer
    x = Dropout(0.5)(x)  # Dropout for regularization
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)
    
    # Freeze layers in the base model
    for layer in base_model.layers:
        layer.trainable = False
    
    return model

# Reshape landmarks to fit MobileNet's input shape
input_shape = X_train_images.shape[1:]  # Shape of one sample
num_classes = len(np.unique(y_train_images))  # Number of unique labels

# Build the improved MobileNet model
mobilenet_images_model = build_improved_mobilenet(input_shape, num_classes)

# Compile the improved MobileNet model
mobilenet_images_model.compile(optimizer='adam',
                                        loss='sparse_categorical_crossentropy',
                                        metrics=['accuracy'])

# Train the improved MobileNet model
mobilenet_images_model.fit(X_train_images, y_train_images, epochs=10, batch_size=32)


/var/folders/50/kpjf_c_57vd3djg90xhvs7pc0000gn/T/ipykernel_47400/3962387506.py:6: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)


       0/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   16384/17225924 ━━━━━━━━━━━━━━━━━━━━ 1:59 7us/step

   40960/17225924 ━━━━━━━━━━━━━━━━━━━━ 1:19 5us/step

   57344/17225924 ━━━━━━━━━━━━━━━━━━━━ 1:16 4us/step

...

274/285 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7670 - loss: 0.7094

277/285 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7669 - loss: 0.7095

280/285 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7669 - loss: 0.7096

283/285 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7669 - loss: 0.7097

285/285 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.7638 - loss: 0.7183


In [5]:
# Evaluate the model on the test data
test_loss, test_accuracy = mobilenet_images_model.evaluate(X_test_images, y_test_images)
print(f'Test accuracy: {test_accuracy*100}')


 1/61 ━━━━━━━━━━━━━━━━━━━━ 14s 241ms/step - accuracy: 0.8438 - loss: 0.7451

 4/61 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8164 - loss: 0.7350  

 7/61 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8178 - loss: 0.6955

10/61 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8175 - loss: 0.6706

13/61 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8156 - loss: 0.6580

...

55/61 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8037 - loss: 0.6478

58/61 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8034 - loss: 0.6476

61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8030 - loss: 0.6476

61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7959 - loss: 0.6440


Test accuracy: 79.58974242210388


#### Saving model for persistance

In [6]:
mobilenet_images_model.save('../../../trained-models/mobilenet_images_model.h5')

